# Lesson 1: Tensors & shapes

**Why:** every part of a GPT does the same thing: it takes grids of numbers (**tensors**), multiplies them, and reshapes them. Most bugs in this kind of code are **shape bugs**, where two grids don't fit together. If you can read a shape, you can read the GPT.

**Picture:** a tensor's shape tells you how the numbers are arranged. Read it from left (outermost) to right (innermost):

```
scalar   5                          shape ()          one number
vector   [1, 2, 3]                  shape (3,)        a list
matrix   [[1, 2, 3],                shape (2, 3)      2 rows, 3 columns
          [4, 5, 6]]
3-D      2 stacked 3×4 matrices     shape (2, 3, 4)   = (B, T, C) in our GPT
```

In our GPT, **(B, T, C)** means: **B** chunks of text, each **T** characters long, with each character described by **C** numbers.

**How this lesson works:** each code cell has a **Predict** question above it. Write your guess in the chat (or just in your head), then run the cell with **Shift + Enter** and compare.

## 1. Making tensors

`torch.tensor(...)` turns a Python list into a tensor. `.shape` gives its size along each dimension, and `.ndim` gives how many dimensions it has.

**Predict:** what shape and `ndim` will each of the three print?

In [ ]:
import torch

torch.manual_seed(42)  # makes "random" numbers the same on every run

scalar = torch.tensor(5.0)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0]])

for name, t in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(f"{name:6}  shape={tuple(t.shape)}  ndim={t.ndim}")

Now a 3-D tensor in our (B, T, C) layout. `torch.arange(24)` makes the numbers 0 to 23, and `.reshape(2, 3, 4)` arranges them into 2 blocks of 3 rows with 4 numbers each.

**Predict:** in which block, row and position will the number **17** appear? (Count from 0.)

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)   # (B=2, T=3, C=4)

print("shape:", x.shape)
print(x)

## 2. Picking parts of a tensor (indexing)

Two rules:
- **A number** picks one item and **removes** that dimension.
- **`:`** means "take all of them" and **keeps** that dimension.

`-1` means "the last one". `...` means "all the dimensions I didn't mention".

**Predict:** the shape of each result, and the exact value of `x[1, 2, 3]`.

In [ ]:
print("x[0]         shape:", x[0].shape)          # first chunk of text
print("x[0, 1]      value:", x[0, 1])             # first chunk, second character
print("x[1, 2, 3]   value:", x[1, 2, 3])          # a single number
print("x[:, -1]     shape:", x[:, -1].shape)      # last character of every chunk
print("x[..., 0]    value:")
print(x[..., 0])                                  # first channel of every character

## 3. Math on whole tensors, and broadcasting

Math on tensors works **position by position**: no loops needed.

When shapes differ, PyTorch **broadcasts**. It lines the shapes up **from the right**. Each pair of sizes must be **equal**, or **one of them must be 1** (or missing). A size of 1 gets stretched (copied) to fit.

```
a    (2, 3)          a    (2, 3)          a    (2, 3)
row     (3,)  ✓      col  (2, 1)  ✓      bad     (2,)  ✗  (3 vs 2)
```

**Predict:** the numbers in `a + row` and in `a + col`.

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])        # (2, 3)
row = torch.tensor([100.0, 200.0, 300.0])  # (3,)
col = torch.tensor([[10.0],
                    [20.0]])               # (2, 1)

print("a * 10 =\n", a * 10)
print("a + row =\n", a + row)
print("a + col =\n", a + col)

**Predict:** will `a + bad` work, or give an error? Use the right-alignment rule.

In [ ]:
bad = torch.tensor([1.0, 2.0])   # (2,)

try:
    print(a + bad)
except RuntimeError as err:
    print("Error:", str(err).splitlines()[0])

*Why this matters later:* in Lesson 8 we add **position** information of shape (T, C) to **token** information of shape (B, T, C). Broadcasting copies the same position information onto every chunk in the batch.

## 4. Adding up along one dimension

`sum` and `mean` can work on everything, or along **one dimension** (`dim`). **The dimension you name disappears.** With `keepdim=True` it stays as size 1 instead.

For `a` (2 rows, 3 columns): `dim=0` runs **down** the rows (one result per column), and `dim=1` runs **across** each row (one result per row).

**Predict:** the value and shape of each line.

In [ ]:
print("a.sum()                     :", a.sum())
print("a.sum(dim=0)                :", a.sum(dim=0))
print("a.sum(dim=1)                :", a.sum(dim=1))
print("a.mean(dim=1, keepdim=True) :")
print(a.mean(dim=1, keepdim=True))

The same rule works in 3-D. (`.float()` turns the whole numbers from `arange` into decimals, because `mean` needs decimals.)

**Predict:** the two shapes. `x` is (2, 3, 4).

In [ ]:
print("x.float().mean(dim=-1) shape:", x.float().mean(dim=-1).shape)
print("x.sum(dim=1)           shape:", x.sum(dim=1).shape)

## 5. Matrix multiplication `@`: the GPT's main operation

Each number in the result = **one row** of the left grid combined with **one column** of the right grid: multiply the pairs, then add them up.

```
(n, k) @ (k, m)  ->  (n, m)        the inner sizes (k) must match, and they disappear
(2, 3) @ (3, 2)  ->  (2, 2)
```

**Predict:** `out[0, 0]` is row 0 of `A` = [1, 2, 3] combined with column 0 of `W` = [1, 0, 1]. What number is it? Can you work out the whole 2×2 result?

In [ ]:
A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])   # (2, 3)
W = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]])        # (3, 2)

out = A @ W                           # (2, 2)
print("shape:", out.shape)
print(out)

by_hand = (A[0] * W[:, 0]).sum()      # row 0 times column 0, then add up
print("out[0, 0] by hand:", by_hand)

`.T` **transposes** a matrix: rows become columns. A (2, 3) matrix becomes (3, 2).

**Predict:** will `A @ A` work? What shape is `A @ A.T`?

In [ ]:
try:
    print(A @ A)
except RuntimeError as err:
    print("A @ A   -> Error:", str(err).splitlines()[0])

print("A.T     shape:", A.T.shape)
print("A @ A.T shape:", (A @ A.T).shape)

## 6. Batched matrix multiply: all tokens at once

This is exactly how the GPT works. We have B chunks × T characters, each described by C numbers. **One** weight grid of shape (C, 5) converts **every** character's C numbers into 5 new numbers, all in a single `@`.

```
tokens (B, T, C) @ W_proj (C, 5)  ->  (B, T, 5)
```

(`torch.randn` fills a tensor with random numbers around 0. Their exact values don't matter here, only the shapes.)

**Predict:** the shape of `out`.

In [ ]:
B, T, C = 2, 3, 4
tokens = torch.randn(B, T, C)   # (B, T, C)
W_proj = torch.randn(C, 5)      # (C, 5)

out = tokens @ W_proj           # (B, T, 5)
print("out shape:", out.shape)

With two 3-D tensors, `@` multiplies each chunk in the batch separately. `.transpose(-2, -1)` swaps the last two dimensions (the 3-D version of `.T`).

**Predict:** both shapes. *(Coming in Lesson 8: that last T × T grid becomes the "who pays attention to whom" table.)*

In [ ]:
tokens_t = tokens.transpose(-2, -1)   # (B, C, T)
scores = tokens @ tokens_t            # (B, T, C) @ (B, C, T) -> (B, T, T)

print("tokens_t shape:", tokens_t.shape)
print("scores   shape:", scores.shape)

## 7. Reshaping with `.view`

`.view` keeps the same numbers in the same order and just changes how they're grouped. Here it flattens (B, T, 5) into (B·T, 5), a flat list of rows, one per character. (Lesson 6 needs this exact move for the loss.)

When flattening, chunk `b` at position `t` ends up at row `b * T + t`.

**Predict:** the shape of `flat`, and whether `flat[4]` is the same as `out[1, 1]` (True or False). Remember T = 3.

In [ ]:
flat = out.view(B * T, 5)        # (B*T, 5)

print("flat shape:", flat.shape)
print("flat[4] == out[1, 1]:", torch.equal(flat[4], out[1, 1]))

## 8. Final check

In [ ]:
assert x[:, -1].shape == (2, 4)
assert (a + col).shape == (2, 3)
assert a.sum(dim=1).shape == (2,)
assert (A @ W).shape == (2, 2)
assert (tokens @ W_proj).shape == (B, T, 5)
assert flat.shape == (B * T, 5)

print("All shape checks passed. Lesson 1 complete.")

## Wrap-up questions (answer in the chat)

1. `x` has shape **(4, 8, 32)**. What are the shapes of `x[0]`, `x[:, 0]`, `x.sum(dim=1)` and `x.mean(dim=-1, keepdim=True)`?
2. In our GPT, `tokens` has shape **(B, T, C) = (4, 8, 32)** and the final weight grid has shape **(32, 85)**. What shape does `tokens @ weights` give, and what do you think the **85** stands for? (Hint: Lesson 0.)
3. **Find the bug:** `q` and `k` both have shape **(2, 3, 4)**. `q @ k` crashes. Why? What one change fixes it, and what shape do you get then?